# Lesson 07 - Suunnittelumallin suunnittelu

Tämä muistikirja havainnollistaa **suunnittelumallin suunnittelua** tekoälyagenttien avulla Microsoft Agent Frameworkilla.  
Opit, kuinka monimutkaiset matkatoiveet jaetaan jäsenneltyihin alitehtäviin, annetaan specialistiagenteille ja toteutetaan lopullinen suunnitelma — kaikki käyttäen Pydantic-mallien tuottamaa jäsenneltyä tulosta.


## Asennus


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Tehtävän pilkkominen

Tehtävän pilkkominen on suunnittelumallin ydin. Sen sijaan, että pyydettäisiin yksittäistä agenttia käsittelemään monimutkainen pyyntö kokonaisuutena, jaamme ongelman pienempiin, selkeästi määriteltyihin **osatehtäviin**. Jokainen osatehtävä annetaan erikoistuneelle agentille (esim. lennot, hotellit, aktiviteetit) selkeillä prioriteeteilla ja riippuvuusjärjestyksellä.

Tämä lähestymistapa tarjoaa useita etuja:
- **Selkeys**: jokaisella osatehtävällä on yksi vastuualuensa.
- **Rinnakkaisuus**: riippumattomat osatehtävät voivat toimia samanaikaisesti.
- **Luotettavuus**: virheet rajoittuvat yksittäisiin osatehtäviin.
- **Budjetin seuranta**: kustannukset arvioidaan osatehtämittäin ja kootaan yhteen.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Suunnittelutoimijan luominen jäsennellyllä tulostuksella

Suunnittelutoimija toimii **vastaanoton koordinaattorina**. Kun sille annetaan korkean tason matkatoive, se tuottaa jäsennellyn `TravelPlan`-suunnitelman — jakamalla pyynnön alatehtäviin, asettamalla prioriteetteja ja tunnistamalla riippuvuuksia, jotta concierge tai toteutuskerros voi suorittaa työn.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Suunnitelman toteuttaminen erikoistyökaluilla

Kun vastaanottovirkailija on laatinut rakenteellisen suunnitelman, **concierge-agentti** toteuttaa sen.  
Jokainen erikoistyökalu hoitaa yhden alitehtäväluokan (lennot, hotellit, aktiviteetit). Concierge käy läpi suunnitelman alitehtävät riippuvuusjärjestyksessä ja lähettää jokaisen oikeaan työkaluun.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Yhteenveto

Tässä oppitunnissa opit **Suunnittelumallin** tekoälyagentteja varten:

1. **Tehtävän jakaminen** — Vastaanoton suunnitteluagentti pilkkoo monimutkaisen matkapyynnön rakenteellisiksi alat tehtäviksi käyttäen Pydantic-malleja, määrittäen kullekin erikoisagentille prioriteetit ja riippuvuudet.
2. **Rakenteellinen tuloste** — Antamalla `response_format` agentti palauttaa validoidun `TravelPlan`-objektin vapaamuotoisen tekstin sijaan, tehden jatkokäsittelystä luotettavaa.
3. **Suunnitelman toteutus** — Concierge-agentti käy läpi alat tehtävät erikoistyökaluilla (`book_flight`, `reserve_hotel`, `book_activity`) toteuttaakseen suunnitelman ja raportoidakseen tulokset.

Tämä malli erottaa *mitä tehdä* (suunnittelu) ja *miten tehdä* (toteutus), tehden agenteista modulaarisempia, testattavampia ja laajennettavampia.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:  
Tämä asiakirja on käännetty tekoälypohjaisen käännöspalvelun [Co-op Translator](https://github.com/Azure/co-op-translator) avulla. Pyrimme tarkkuuteen, mutta huomioithan, että konekäännöksissä voi esiintyä virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäisellä kielellä on päätösvaltainen lähde. Tärkeissä tiedoissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
